# Pertemuan 4 — Mendiagnosis Kualitas Data
### Workbook belajar mandiri

**Exploratory Data Analysis (KMTI21133)**  
**Sub-CPMK4:** Mahasiswa mampu mendiagnosis masalah kualitas data: *missing values*, duplikasi, inkonsistensi, dan kesalahan tipe data (C4).

---

> **Dosen berhalangan hadir pada pertemuan 4 dan 5.** Kelas didampingi asisten, dan workbook ini dirancang agar
> dapat Anda selesaikan sendiri — di kelas bersama asisten, atau sepenuhnya mandiri di luar kelas.

### Cara memakai workbook ini

| | |
|---|---|
| **Perkiraan waktu** | 150 menit. Bagian 1–5 sekitar 75 menit, Bagian 6–9 sekitar 50 menit, Bagian 10 sekitar 25 menit. |
| **Sel penjelasan** | Dibaca. Tidak ada yang perlu dijalankan. |
| **Sel CONTOH** | Dijalankan apa adanya, lalu amati keluarannya. |
| **Sel TUGAS** | Anda isi sendiri, lalu jalankan sel `cek(...)` di bawahnya. |
| **Umpan balik** | Sel `cek(...)` akan menjawab **BENAR** atau **BELUM** beserta petunjuk. Jawaban tidak ditampilkan, jadi silakan mencoba sebanyak yang Anda perlu. |
| **Kalau mentok** | Lihat Bagian *Kalau Anda mentok* di akhir workbook sebelum bertanya. |

### Yang Anda hasilkan di akhir sesi

1. **Profil kualitas data** untuk dataset latihan (Bagian 9).
2. **Profil kualitas data awal untuk dataset proyek tim** (Bagian 10) — ini bagian pertama dari **M2** yang dikumpulkan pada pertemuan 5.

> **Jangan memperbaiki apa pun hari ini.** Pertemuan 4 hanya mendiagnosis. Perbaikan dikerjakan pada pertemuan 5.
> Memperbaiki sebelum mendiagnosis adalah cara tercepat menghilangkan bukti.

---
## Bagian 0 — Persiapan

Dataset: **`transaksi_koperasi_2025.csv`** — catatan transaksi koperasi mahasiswa, Agustus–November 2025.

> Seluruh isinya **sintetis**. Tidak ada orang nyata di dalamnya. Namun cacat datanya dirancang menyerupai
> data operasional sungguhan, termasuk jenis cacat yang paling sering luput dari perhatian.

In [1]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

FILE = 'transaksi_koperasi_2025.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)
print('Berhasil dimuat:', df.shape)

Berhasil dimuat: (1543, 16)


### Alat pemeriksa jawaban
Jalankan sel berikut satu kali. Setelah itu Anda bisa memanggil `cek('T1_baris', jawaban_saya)` kapan saja.

In [2]:
_K = {
    'T1_baris': '50a0a04f2d',
    'T1_kolom': 'b17ef6d19c',
    'T2_kosong': '3a481e7283',
    'T2_kolom': 'ef2d127de3',
    'T3_rating': '0e17daca5f',
    'T4_jasa': '43b87f618c',
    'T5_dup': 'c2356069e9',
    'T5_dupid': '8241649609',
    'T6_kat': '4ec9599fc2',
    'T6_katbersih': 'ef2d127de3',
    'T7_rp': '1e472b39b1',
    'T8_jumlah': '35135aaa6c',
    'T8_diskon': '19581e27de',
    'T9_anggota': 'aae0212936',
    'T9_strip': '7c252ab334',
    'T10_bisnis': '44c8031cb0',
    'T12_bulan': '4b227777d4',
    'T12_rusak': '48ce32e8ec'
}

_P = {
    'T1_baris': 'Pakai df.shape[0] atau len(df).',
    'T1_kolom': 'Pakai df.shape[1].',
    'T2_kosong': 'df.isna().sum() memberi per kolom; jumlahkan lagi dengan .sum().',
    'T2_kolom': 'Hitung berapa kolom yang nilai isna().sum()-nya lebih dari 0.',
    'T3_rating': 'Bandingkan kolom rating_layanan dengan -999, lalu jumlahkan.',
    'T4_jasa': 'Saring baris berkategori Jasa (rapikan dulu spasi dan hurufnya), lalu .isna().mean()*100 pada berat_kg. Bulatkan 1 desimal.',
    'T5_dup': 'df.duplicated().sum() menghitung baris yang SELURUH kolomnya sama.',
    'T5_dupid': 'Pakai df[\'id_transaksi\'].duplicated(keep=False).sum() agar kedua pasangannya ikut terhitung.',
    'T6_kat': 'Hitung nunique() pada kolom kategori apa adanya, tanpa dirapikan.',
    'T6_katbersih': 'Rapikan: .str.strip().str.title(), lalu satukan \'Minumam\'->\'Minuman\' dan \'Alat Tulis\'->\'Atk\'.',
    'T7_rp': 'Ubah ke teks dengan .astype(str), lalu .str.startswith(\'Rp\').',
    'T8_jumlah': 'Hitung baris dengan jumlah kurang dari atau sama dengan nol.',
    'T8_diskon': 'Gabungkan dua kondisi dengan | dan bungkus masing-masing dengan tanda kurung.',
    'T9_anggota': 'nunique() pada id_anggota apa adanya.',
    'T9_strip': 'nunique() setelah .str.strip().',
    'T10_bisnis': 'Kerjakan hanya pada baris yang total_bayar-nya sudah berupa angka. Bandingkan dengan (jumlah x harga x (1 - diskon/100)).round().',
    'T12_bulan': 'Setelah konversi yang benar, pakai .dt.month.nunique(). Data ini hanya mencakup beberapa bulan saja.',
    'T12_rusak': 'Bandingkan hasil konversi yang benar dengan hasil pd.to_datetime(..., format=\'mixed\', dayfirst=True) memakai !=, lalu jumlahkan.'
}

def _h(x):
    return hashlib.sha256(str(x).strip().lower().encode()).hexdigest()[:10]

def cek(tugas, jawaban):
    """Periksa jawaban Anda tanpa menampilkan kuncinya."""
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas); return
    if _h(jawaban) == _K[tugas]:
        print(f'BENAR   {tugas} = {jawaban}')
    else:
        print(f'BELUM   {tugas} = {jawaban}')
        print('        Petunjuk:', _P.get(tugas, '-'))

def skor():
    """Tidak melacak apa pun; sekadar pengingat daftar tugas."""
    print('Daftar tugas:', ', '.join(sorted(_K)))

print('Alat pemeriksa siap. Contoh pemakaian: cek("T1_baris", 1000)')

Alat pemeriksa siap. Contoh pemakaian: cek("T1_baris", 1000)


---
## Bagian 1 — Lima dimensi kualitas data

Sebelum menyentuh kode, kenali apa saja yang sedang kita cari. Kualitas data biasanya dipecah menjadi lima dimensi.

| Dimensi | Pertanyaannya | Contoh masalah |
|---|---|---|
| **Kelengkapan** | Apakah ada nilai yang hilang? | Kolom kosong, atau placeholder seperti `-999` dan `N/A` |
| **Keunikan** | Apakah ada yang tercatat dua kali? | Baris duplikat persis maupun nyaris sama |
| **Konsistensi** | Apakah hal yang sama ditulis dengan cara yang sama? | `ATK`, `Atk`, ` ATK `, dan `Alat Tulis` sebagai empat kategori berbeda |
| **Validitas** | Apakah nilainya mungkin terjadi? | Jumlah barang negatif, diskon 150%, tanggal di masa depan |
| **Akurasi** | Apakah nilainya benar? | Total bayar tidak sama dengan jumlah × harga |

Urutan pemeriksaannya mengikuti urutan tabel ini, karena masalah di baris atas sering menyebabkan
salah tafsir pada baris bawah. Misalnya, menghitung rata-rata rating tanpa menyadari adanya `-999`
akan menghasilkan angka yang masuk akal secara tampilan tetapi salah total.

### Satu prinsip yang menentukan segalanya

> **Data yang kelihatan rapi belum tentu benar, dan pandas tidak akan memberi tahu Anda.**

`df.head()` akan tampil cantik untuk data seburuk apa pun. Satu-satunya cara mengetahui kondisi data
adalah memeriksanya satu dimensi demi satu dimensi, dengan sengaja.

---
## Bagian 2 — Potret awal

Tujuh perintah ini adalah hal pertama yang dijalankan pada dataset apa pun, selamanya.

In [3]:
# CONTOH 2.1 — tampilan permukaan
df.head(5)

,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
0,TRX251385,2025-10-29 11:19:00,AGT1150,Mi Instan Rebus,Makanan,porsi,6,10000,0,60000,QRIS,Rina,392000.0,2.445,3,NaN
1,TRX251333,2025-09-29 11:46:00,AGT1259,Teh Botol,Minuman,botol,3,6000,0,Rp 18.000,Saldo Koperasi,Dimas,152000.0,0.583,5,Pelanggan tetap
2,TRX251278,2025-10-25 13:18:00,AGT1145,Map Plastik,ATK,pcs,4,4000,0,16000,Saldo Koperasi,Bagus,127000.0,2.181,5,Pelanggan tetap
3,TRX251304,2025-09-09 13:45:00,AGT1240,Fotokopi A4,Alat Tulis,pcs,1,500,0,250,Tunai,Rina,1008000.0,NaN,5,NaN
4,TRX251364,2025-08-09 10:48:00,AGT1191,Map Plastik,ATK,pcs,5,4000,5,19000,QRIS,Rina,139000.0,2.376,4,Pelanggan tetap


In [4]:
# CONTOH 2.2 — ukuran dan tipe
print('Ukuran:', df.shape)
print()
df.info()

Ukuran: (1543, 16)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1543 entries, 0 to 1542
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id_transaksi       1536 non-null   object 
 1   tanggal_transaksi  1543 non-null   object 
 2   id_anggota         1543 non-null   object 
 3   nama_produk        1543 non-null   object 
 4   kategori           1543 non-null   object 
 5   satuan             1543 non-null   object 
 6   jumlah             1543 non-null   int64  
 7   harga_satuan       1543 non-null   int64  
 8   diskon_persen      1543 non-null   int64  
 9   total_bayar        1543 non-null   object 
 10  metode_bayar       1534 non-null   object 
 11  kasir              1543 non-null   object 
 12  saldo_simpanan     1384 non-null   float64
 13  berat_kg           1192 non-null   float64
 14  rating_layanan     1543 non-null   int64  
 15  catatan_kasir      499 non-null    object 
dtypes: f

> **Amati kolom `Dtype` di atas.** Tiga hal yang ganjil:
>
> - `tanggal_transaksi` bertipe `object`, padahal isinya tanggal.
> - `total_bayar` bertipe `object`, padahal isinya uang.
> - `kategori` dan `metode_bayar` bertipe `object` — wajar untuk teks, tetapi perlu dicek isinya.
>
> Kolom uang yang terbaca sebagai teks hampir selalu berarti ada nilai berformat di dalamnya,
> misalnya `Rp 24.000`. Satu nilai seperti itu sudah cukup membuat seluruh kolom menjadi teks.

In [5]:
# CONTOH 2.3 — ringkasan numerik
df.describe().round(2)

,jumlah,harga_satuan,diskon_persen,saldo_simpanan,berat_kg,rating_layanan
count,1543.00,1543.00,1543.00,1384.00,1192.00,1543.00
mean,4.12,7891.77,1.01,336653.18,1.51,-27.72
std,2.58,6549.15,6.63,360898.81,1.11,175.96
min,-8.00,500.00,-20.00,10000.00,0.05,-999.00
25%,2.00,4000.00,0.00,120000.00,0.59,4.00
50%,4.00,6000.00,0.00,226000.00,1.17,4.00
75%,6.00,10000.00,0.00,422000.00,2.26,5.00
max,8.00,35000.00,150.00,3516000.00,4.78,5.00


> **Lihat baris `min` pada `rating_layanan`.** Nilainya `-999`, dan `mean`-nya ikut menjadi negatif.
> Ini bukan kesalahan hitung pandas; pandas memang diminta merata-ratakan angka `-999`.
> Angka seperti ini disebut **placeholder**: tanda data kosong yang ditulis sebagai angka,
> sehingga lolos dari `isna()`.
>
> Perhatikan juga `min` pada `jumlah` dan `max` pada `diskon_persen`.

In [6]:
# CONTOH 2.4 — ringkasan kolom teks
df.describe(include='object').T

,count,unique,top,freq
id_transaksi,1536,1493,TRX251348,3
tanggal_transaksi,1543,1495,23/10/2025 09:52,3
id_anggota,1543,318,AGT1041,16
nama_produk,1543,18,Roti Isi,144
kategori,1543,18,Makanan,469
satuan,1543,5,porsi,508
total_bayar,1543,220,12000,79
metode_bayar,1534,11,Tunai,541
kasir,1543,8,Bagus,416
catatan_kasir,499,4,Pelanggan tetap,180


### TUGAS 1 — ukuran dataset

In [15]:
# TUGAS 1
t1_baris = df.shape[0]   # ganti dengan jumlah baris
t1_kolom = df.shape[1]  # ganti dengan jumlah kolom

cek('T1_baris', t1_baris)
cek('T1_kolom', t1_kolom)

BENAR   T1_baris = 1543
BENAR   T1_kolom = 16


---
## Bagian 3 — Kelengkapan: nilai kosong dan penyamarannya

Ada dua macam nilai kosong, dan hanya satu yang terlihat oleh `isna()`.

| Jenis | Contoh | Terdeteksi `isna()`? |
|---|---|---|
| Kosong sejati | `NaN`, sel kosong di CSV | Ya |
| Placeholder angka | `-999`, `0` yang berarti tidak diisi, `9999` | **Tidak** |
| Placeholder teks | `N/A`, `-`, `null`, `tidak diketahui`, `?` | **Tidak** |

Placeholder berbahaya justru karena ia terlihat seperti data sungguhan.

In [9]:
# CONTOH 3.1 — kosong sejati
kosong = df.isna().sum()
kosong[kosong > 0].to_frame('jumlah_kosong').assign(
    persen=lambda d: (d.jumlah_kosong / len(df) * 100).round(1))

,jumlah_kosong,persen
id_transaksi,7,0.5
metode_bayar,9,0.6
saldo_simpanan,159,10.3
berat_kg,351,22.7
catatan_kasir,1044,67.7


In [10]:
# CONTOH 3.2 — berburu placeholder teks
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

for kol in df.select_dtypes(include='object').columns:
    nilai = df[kol].astype(str).str.strip().str.lower()
    ketemu = nilai[nilai.isin(CURIGA)].value_counts()
    if len(ketemu):
        print(f'{kol}:')
        print(ketemu.to_string(), '\n')

metode_bayar:
metode_bayar
-                  13
tidak diketahui     6 



In [12]:
# CONTOH 3.3 — berburu placeholder angka
# Caranya: lihat nilai ekstrem pada tiap kolom numerik
for kol in df.select_dtypes(include='number').columns:
    s = df[kol]
    print(f'{kol:18s} min={s.min():12,.1f}  max={s.max():14,.1f}  '
          f'nilai terkecil unik: {sorted(s.dropna().unique())[:3]}')

jumlah             min=        -8.0  max=           8.0  nilai terkecil unik: [np.int64(-8), np.int64(-6), np.int64(-5)]
harga_satuan       min=       500.0  max=      35,000.0  nilai terkecil unik: [np.int64(500), np.int64(2000), np.int64(3500)]
diskon_persen      min=       -20.0  max=         150.0  nilai terkecil unik: [np.int64(-20), np.int64(0), np.int64(5)]
saldo_simpanan     min=    10,000.0  max=   3,516,000.0  nilai terkecil unik: [np.float64(10000.0), np.float64(16000.0), np.float64(18000.0)]
berat_kg           min=         0.1  max=           4.8  nilai terkecil unik: [np.float64(0.05), np.float64(0.053), np.float64(0.055)]
rating_layanan     min=      -999.0  max=           5.0  nilai terkecil unik: [np.int64(-999), np.int64(2), np.int64(3)]


> **Pola yang perlu dihafal.** Placeholder angka hampir selalu berupa bilangan yang mencolok:
> `-1`, `-999`, `0`, `9999`, `99999`, atau tahun `1900`. Kalau `min` atau `max` sebuah kolom terasa aneh,
> periksa nilai uniknya sebelum menganggapnya data sungguhan.

### TUGAS 2 — kelengkapan

In [14]:
# TUGAS 2
t2_kosong = df.isna().sum().sum()  # TOTAL sel kosong di seluruh dataframe
t2_kolom  = (df.isna().sum() > 0).sum()  # BERAPA KOLOM yang punya setidaknya satu nilai kosong

cek('T2_kosong', t2_kosong)
cek('T2_kolom', t2_kolom)

BENAR   T2_kosong = 1570
BENAR   T2_kolom = 5


### TUGAS 3 — placeholder angka

In [16]:
# TUGAS 3
t3_rating = (df['rating_layanan'] == -999).sum()   # berapa baris yang rating_layanan-nya bernilai -999

cek('T3_rating', t3_rating)

BENAR   T3_rating = 49


---
## Bagian 4 — Mengapa data itu kosong: MCAR, MAR, MNAR

Menemukan nilai kosong baru separuh pekerjaan. Separuh lainnya adalah menebak **mengapa** kosong,
karena alasannya menentukan apakah Anda boleh menghapus baris, boleh mengisi, atau tidak boleh keduanya.

| Istilah | Artinya | Contoh | Akibatnya |
|---|---|---|---|
| **MCAR** | Kosong betul-betul acak | Kertas struk sobek | Menghapus baris aman, hanya kehilangan ukuran sampel |
| **MAR** | Kosong bergantung pada kolom lain **yang terlihat** | Berat barang kosong untuk produk non-fisik | Bisa ditangani; isi per kelompok, atau akui bahwa kolom itu memang tidak berlaku |
| **MNAR** | Kosong bergantung pada **nilainya sendiri** yang tidak terlihat | Saldo besar sengaja tidak ditampilkan | Paling berbahaya; mengisi rata-rata akan menyesatkan |

> **Hal terpenting di bagian ini:** MAR dapat dibuktikan dari data. MCAR dan MNAR **tidak bisa dibedakan**
> hanya dengan melihat data, karena nilai yang hilang tidak dapat diperiksa. Membedakannya memerlukan
> pengetahuan tentang bagaimana data itu dikumpulkan.

In [17]:
# CONTOH 4.1 — alat bantu: apakah kekosongan terkait kolom lain?
def pola_kosong(data, kolom_kosong, kolom_pembanding):
    """Persentase kekosongan kolom_kosong untuk tiap nilai kolom_pembanding."""
    return (data.groupby(kolom_pembanding)[kolom_kosong]
                .apply(lambda s: round(s.isna().mean() * 100, 1))
                .sort_values(ascending=False)
                .to_frame('persen_kosong'))

# rapikan kategori lebih dulu supaya pengelompokannya bermakna
kat_rapi = (df['kategori'].astype(str).str.strip().str.title()
              .replace({'Minumam': 'Minuman', 'Alat Tulis': 'Atk'}))
df_bantu = df.assign(kategori_rapi=kat_rapi)

pola_kosong(df_bantu, 'berat_kg', 'kategori_rapi')

,persen_kosong
kategori_rapi,
Jasa,100.0
Pulsa,100.0
Atk,16.1
Makanan,0.0
Minuman,0.0


> **Ini MAR, dan buktinya jelas.** `berat_kg` kosong 100% untuk Jasa dan Pulsa — keduanya memang tidak
> punya berat. Kekosongannya bergantung pada `kategori`, kolom yang bisa kita lihat.
>
> Tindakan yang tepat **bukan** mengisi dengan rata-rata, melainkan mengakui bahwa kolom ini hanya
> berlaku untuk barang fisik. Mengisi berat Pulsa dengan 0,3 kg akan menciptakan data palsu.

In [18]:
# CONTOH 4.2 — bandingkan dengan kolom lain
pola_kosong(df_bantu, 'catatan_kasir', 'kategori_rapi')

,persen_kosong
kategori_rapi,
Pulsa,74.3
Jasa,70.2
Makanan,69.7
Minuman,67.1
Atk,61.0


> **Ini tampak MCAR.** Persentasenya mirip di semua kategori. Tidak ada kolom teramati yang menjelaskan
> kekosongannya, jadi menghapus atau mengabaikan kolom ini relatif aman.
>
> Catatan: sebagian kekosongan ini sebenarnya berasal dari sel yang memang dibiarkan kosong oleh kasir
> (teks kosong), yang oleh `read_csv` dibaca menjadi `NaN`. Teks kosong dan data hilang adalah dua hal
> berbeda, tetapi CSV tidak bisa membedakannya. Ini keterbatasan format, bukan kesalahan Anda.

In [19]:
# CONTOH 4.3 — kasus yang menipu
for kol in ['kategori_rapi', 'kasir', 'metode_bayar']:
    print(f'--- saldo_simpanan kosong menurut {kol} ---')
    print(pola_kosong(df_bantu, 'saldo_simpanan', kol).head(5).to_string(), '\n')

--- saldo_simpanan kosong menurut kategori_rapi ---
               persen_kosong
kategori_rapi               
Minuman                 12.3
Jasa                    11.9
Pulsa                    9.5
Atk                      9.3
Makanan                  9.1 

--- saldo_simpanan kosong menurut kasir ---
       persen_kosong
kasir               
dimas           25.0
Bagus           11.3
Sari            11.3
Rina             9.8
Dimas            8.8 

--- saldo_simpanan kosong menurut metode_bayar ---
                persen_kosong
metode_bayar                 
tunai                    20.0
saldo koperasi           20.0
Saldo Koperasi           12.1
QRIS                     10.4
Tunai                    10.4 



In [20]:
# CONTOH 4.4 — dan pembandingan rata-rata kolom numerik
m = df['saldo_simpanan'].isna()
banding = pd.DataFrame({
    'saat_kosong': df.loc[m, ['jumlah', 'harga_satuan', 'diskon_persen']].mean().round(2),
    'saat_terisi': df.loc[~m, ['jumlah', 'harga_satuan', 'diskon_persen']].mean().round(2),
})
print(banding.to_string())
print()
print('Saldo yang TERISI — median:', f"{df['saldo_simpanan'].median():,.0f}",
      '| maksimum:', f"{df['saldo_simpanan'].max():,.0f}")

               saat_kosong  saat_terisi
jumlah                4.48         4.08
harga_satuan       7606.92      7924.49
diskon_persen         0.66         1.05

Saldo yang TERISI — median: 226,000 | maksimum: 3,516,000


> **Baca pelan-pelan, ini inti pertemuan hari ini.**
>
> Setiap pemeriksaan di atas mengatakan kekosongan `saldo_simpanan` tidak terkait kolom mana pun.
> Secara statistik, kolom ini **tampak MCAR**, dan Anda akan tergoda mengisinya dengan median.
>
> Sekarang perhatikan satu hal yang tidak muncul di pemeriksaan mana pun: saldo tertinggi yang
> *masih terlihat* hanya sekitar 3,5 juta. Untuk koperasi dengan ratusan anggota, tidak adanya satu pun
> saldo besar adalah hal yang ganjil. Dugaan yang masuk akal: anggota bersaldo besar memilih tidak
> menampilkan saldonya.
>
> Kalau dugaan itu benar, kolom ini **MNAR**, dan mengisinya dengan median akan membuat rata-rata saldo
> koperasi tampak jauh lebih kecil daripada kenyataannya. Tidak ada satu pun uji statistik yang bisa
> memastikan hal ini. Yang bisa memastikannya hanya satu: **bertanya kepada pemilik data**.

### TUGAS 4 — membuktikan MAR

In [21]:
# TUGAS 4
# Berapa persen baris berkategori Jasa yang berat_kg-nya kosong?
# Bulatkan satu desimal. Gunakan kategori yang sudah dirapikan.
t4_jasa = (
    df['kategori']
    .astype(str)
    .str.strip()
    .str.title()
    .replace({'Minumam': 'Minuman', 'Alat Tulis': 'Atk'})
)
df_bantu = df.assign(kategori_rapi=kat_rapi)
t4_jasa = round(
    df_bantu[df_bantu['kategori_rapi'] == 'Jasa']['berat_kg'].isna().mean()
    * 100,
    1,
)

cek('T4_jasa', t4_jasa)

BENAR   T4_jasa = 100.0


**Pertanyaan renungan (tulis jawabannya di sini, tidak ada sel cek):**

Untuk dataset proyek tim Anda, sebutkan satu kolom yang Anda duga MNAR, dan jelaskan
mengapa dugaan itu tidak bisa dibuktikan dari datanya sendiri.

_(tulis di sini)_

---
## Bagian 5 — Keunikan: duplikat persis dan duplikat nyaris

Duplikat persis mudah ditemukan. Yang merepotkan adalah duplikat nyaris: baris yang merujuk kejadian
yang sama tetapi berbeda sedikit, misalnya beda huruf besar-kecil atau satu kolom kosong.

In [22]:
# CONTOH 5.1 — duplikat persis
print('Baris yang seluruh kolomnya sama:', df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values('id_transaksi').head(6)

Baris yang seluruh kolomnya sama: 24


,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
1208,TRX250149,2025-10-11 08:11:00,AGT1071,Buku Tulis 38lbr,ATK,pcs,2,5500,0,11000,Qris,Dimas,85000.0,NaN,4,Pelanggan tetap
1109,TRX250149,2025-10-11 08:11:00,AGT1071,Buku Tulis 38lbr,ATK,pcs,2,5500,0,11000,Qris,Dimas,85000.0,NaN,4,Pelanggan tetap
634,TRX250199,2025-10-31 12:48:00,AGT1041,Paket Data 3GB,Pulsa,voucher,1,35000,0,35000,Tunai,Rina,323000.0,NaN,4,NaN
475,TRX250199,2025-10-31 12:48:00,AGT1041,Paket Data 3GB,Pulsa,voucher,1,35000,0,35000,Tunai,Rina,323000.0,NaN,4,NaN
283,TRX250217,2025-09-20 14:01:00,AGT1128,Nasi Bungkus,Makanan,porsi,4,12000,0,Rp 48.000,QRIS,Dimas,37000.0,1.792,5,NaN
581,TRX250217,2025-09-20 14:01:00,AGT1128,Nasi Bungkus,Makanan,porsi,4,12000,0,Rp 48.000,QRIS,Dimas,37000.0,1.792,5,NaN


In [23]:
# CONTOH 5.2 — duplikat pada kunci utama
# id_transaksi seharusnya unik. Kalau tidak, ada yang salah.
dup_id = df['id_transaksi'].duplicated(keep=False)
print('Baris dengan id_transaksi berulang:', dup_id.sum())
print('id_transaksi kosong             :', df['id_transaksi'].isna().sum())

contoh = df[dup_id].sort_values('id_transaksi').head(4)
contoh[['id_transaksi', 'kasir', 'catatan_kasir', 'total_bayar']]

Baris dengan id_transaksi berulang: 92
id_transaksi kosong             : 7


,id_transaksi,kasir,catatan_kasir,total_bayar
677,TRX250024,rina,NaN,30000
663,TRX250024,Rina,NaN,30000
583,TRX250132,rina,NaN,15000
257,TRX250132,Rina,NaN,15000


> **Perhatikan contoh di atas.** Ada pasangan baris dengan `id_transaksi` yang sama, tetapi nama kasir
> ditulis berbeda kapitalisasinya dan `catatan_kasir`-nya kosong di salah satu baris. `duplicated()`
> biasa tidak menangkapnya, karena bagi pandas kedua baris itu memang berbeda.
>
> Cara menemukannya: cari duplikat pada **kolom kunci saja**, bukan seluruh kolom.

In [24]:
# CONTOH 5.3 — duplikat berdasarkan sebagian kolom
kunci = ['id_transaksi']
print('Duplikat menurut kunci :', df.duplicated(subset=kunci, keep=False).sum())
print('Duplikat seluruh kolom :', df.duplicated(keep=False).sum())
print()
print('Selisihnya adalah duplikat NYARIS, yang justru paling sering luput.')

Duplikat menurut kunci : 92
Duplikat seluruh kolom : 48

Selisihnya adalah duplikat NYARIS, yang justru paling sering luput.


### TUGAS 5 — keunikan

In [25]:
# TUGAS 5
t5_dup   = df.duplicated().sum()   # jumlah baris duplikat menurut df.duplicated() -- perhatikan perilaku default keep='first'
t5_dupid = df['id_transaksi'].duplicated(keep=False).sum()   # jumlah baris yang id_transaksi-nya berulang, dengan keep=False

cek('T5_dup', t5_dup)
cek('T5_dupid', t5_dupid)

BENAR   T5_dup = 24
BENAR   T5_dupid = 92


---
## Bagian 6 — Konsistensi: satu hal, banyak tulisan

Masalah paling umum pada data yang diketik manusia, dan paling sering tidak disadari karena
dampaknya baru muncul saat agregasi.

In [26]:
# CONTOH 6.1 — berapa kategori sebenarnya?
df['kategori'].value_counts(dropna=False)

,count
kategori,
Makanan,469
Minuman,353
ATK,270
Jasa,139
Pulsa,136
Alat Tulis,41
Minumam,22
MINUMAN,20
MAKANAN,20


> Koperasi ini hanya punya lima kategori. Daftar di atas jauh lebih panjang.
> Empat penyebabnya: **huruf besar-kecil**, **spasi di awal atau akhir**, **sinonim**, dan **salah ketik**.

In [27]:
# CONTOH 6.2 — memisahkan keempat penyebab
s = df['kategori'].astype(str)
print('Nilai unik apa adanya            :', s.nunique())
print('Setelah dibuang spasinya         :', s.str.strip().nunique())
print('Setelah diseragamkan hurufnya    :', s.str.strip().str.title().nunique())
print()
print('Sisa yang perlu keputusan manusia:')
print(sorted(s.str.strip().str.title().unique()))

Nilai unik apa adanya            : 18
Setelah dibuang spasinya         : 11
Setelah diseragamkan hurufnya    : 7

Sisa yang perlu keputusan manusia:
['Alat Tulis', 'Atk', 'Jasa', 'Makanan', 'Minumam', 'Minuman', 'Pulsa']


> Dua yang tersisa — `Minumam` dan `Alat Tulis` — **tidak bisa diselesaikan oleh kode**.
> Komputer tidak tahu bahwa `Minumam` adalah salah ketik dari `Minuman`, dan bahwa `Alat Tulis`
> adalah nama lain untuk `ATK`. Keputusan ini harus diambil manusia, lalu **dicatat alasannya**.
>
> Inilah sebabnya pertemuan 5 mewajibkan *cleaning log*.

In [28]:
# CONTOH 6.3 — spasi tersembunyi pada kolom kunci
a = df['id_anggota'].astype(str)
print('id_anggota unik apa adanya :', a.nunique())
print('id_anggota unik setelah strip:', a.str.strip().nunique())
print()
print('Selisih', a.nunique() - a.str.strip().nunique(),
      'anggota hantu — muncul hanya karena ada spasi di akhir teks.')

id_anggota unik apa adanya : 318
id_anggota unik setelah strip: 259

Selisih 59 anggota hantu — muncul hanya karena ada spasi di akhir teks.


> **Dampaknya nyata.** Kalau Anda menghitung jumlah anggota aktif koperasi tanpa membuang spasi,
> laporan Anda akan menyebut puluhan anggota yang sebenarnya tidak ada. Angkanya rapi, tabelnya cantik,
> dan hasilnya salah.

In [29]:
# CONTOH 6.4 — periksa seluruh kolom teks sekaligus
for kol in df.select_dtypes(include='object').columns:
    s = df[kol].astype(str)
    mentah = s.nunique()
    rapi   = s.str.strip().str.lower().nunique()
    if mentah != rapi:
        print(f'{kol:20s} {mentah:4d} -> {rapi:4d}   (selisih {mentah - rapi})')

id_anggota            318 ->  259   (selisih 59)
kategori               18 ->    7   (selisih 11)
metode_bayar           12 ->    7   (selisih 5)
kasir                   8 ->    4   (selisih 4)


### TUGAS 6 — konsistensi kategori

In [30]:
# TUGAS 6
t6_kat       = df['kategori'].nunique()   # jumlah nilai unik kolom kategori APA ADANYA
t6_katbersih = kat_rapi.nunique()   # jumlah nilai unik SETELAH strip, title, dan penyatuan sinonim serta salah ketik

cek('T6_kat', t6_kat)
cek('T6_katbersih', t6_katbersih)

BENAR   T6_kat = 18
BENAR   T6_katbersih = 5


### TUGAS 9 — anggota hantu

In [31]:
# TUGAS 9
t9_anggota = df['id_anggota'].astype(str).nunique()   # nunique id_anggota apa adanya
t9_strip   = df['id_anggota'].astype(str).str.strip().nunique()   # nunique id_anggota setelah strip

cek('T9_anggota', t9_anggota)
cek('T9_strip', t9_strip)

BENAR   T9_anggota = 318
BENAR   T9_strip = 259


---
## Bagian 7 — Tipe data yang salah

In [32]:
# CONTOH 7.1 — uang yang tersimpan sebagai teks
contoh = df['total_bayar'].astype(str)
print('Contoh nilai:', contoh.head(8).tolist())
print()
berformat = contoh.str.startswith('Rp')
print('Nilai berformat "Rp" :', berformat.sum())
print('Nilai berupa angka   :', (~berformat).sum())

Contoh nilai: ['60000', 'Rp 18.000', '16000', '250', '19000', '10500', '35000', '4000']

Nilai berformat "Rp" : 250
Nilai berupa angka   : 1293


> Satu nilai berformat sudah cukup membuat **seluruh kolom** menjadi `object`.
> Selama kolom ini masih teks, `sum()` akan menyambung teks, bukan menjumlahkan uang —
> dan `mean()` akan langsung gagal.

In [33]:
# CONTOH 7.2 — tanggal dengan dua format bercampur
t = df['tanggal_transaksi'].astype(str)
print('Contoh:', t.head(6).tolist())
print()
iso = t.str.match(r'^\d{4}-')
print('Berformat yyyy-mm-dd :', int(iso.sum()))
print('Berformat dd/mm/yyyy :', int((~iso).sum()))

Contoh: ['2025-10-29 11:19:00', '2025-09-29 11:46:00', '2025-10-25 13:18:00', '2025-09-09 13:45:00', '2025-08-09 10:48:00', '2025-08-17 15:39:00']

Berformat yyyy-mm-dd : 1236
Berformat dd/mm/yyyy : 307


### Bagian paling berbahaya di seluruh workbook ini

Cara yang paling sering dipakai orang untuk menangani format campur adalah `format='mixed'`
ditambah `dayfirst=True`, karena di Indonesia tanggal memang ditulis hari lebih dulu.
Jalankan sel berikut dan amati baik-baik.

In [34]:
# CONTOH 7.3 — cara yang tampak benar
cara_umum = pd.to_datetime(t, format='mixed', dayfirst=True, errors='coerce')

print('Gagal dikonversi :', int(cara_umum.isna().sum()))
print('Rentang tanggal  :', cara_umum.min(), 'sampai', cara_umum.max())
print('Bulan yang muncul:', sorted(cara_umum.dt.month.dropna().unique()))

Gagal dikonversi : 0
Rentang tanggal  : 2025-08-01 07:09:00 sampai 2025-11-28 16:25:00
Bulan yang muncul: [np.int32(8), np.int32(9), np.int32(10), np.int32(11)]


> **Nol galat. Tidak ada peringatan apa pun. Dan hasilnya salah.**
>
> Dataset ini berisi transaksi **Agustus sampai November 2025** — empat bulan. Hasil di atas
> menyebar ke seluruh dua belas bulan.
>
> Penyebabnya: `dayfirst=True` ikut diterapkan pada baris yang formatnya sudah `yyyy-mm-dd`.
> Baris `2025-08-01` dibaca sebagai tahun 2025, **hari** 08, **bulan** 01, sehingga menjadi 8 Januari.
> Karena 08 dan 01 keduanya angka yang sah untuk hari maupun bulan, pandas tidak punya alasan mengeluh.
>
> Inilah jenis kesalahan yang paling mahal dalam analisis data: **salah yang tidak menimbulkan galat.**
> Tidak ada yang berwarna merah, tidak ada yang berhenti, dan laporan Anda tetap terbit.

In [35]:
# CONTOH 7.4 — cara yang benar: tangani tiap format secara terpisah
benar = pd.Series(pd.NaT, index=t.index, dtype='datetime64[ns]')
benar[iso]  = pd.to_datetime(t[iso],  format='%Y-%m-%d %H:%M:%S', errors='coerce')
benar[~iso] = pd.to_datetime(t[~iso], format='%d/%m/%Y %H:%M',    errors='coerce')

print('Gagal dikonversi :', int(benar.isna().sum()))
print('Rentang tanggal  :', benar.min(), 'sampai', benar.max())
print('Bulan yang muncul:', sorted(benar.dt.month.dropna().unique()))
print()
print('Baris yang berbeda antara cara umum dan cara benar:', int((cara_umum != benar).sum()))

Gagal dikonversi : 0
Rentang tanggal  : 2025-08-01 07:09:00 sampai 2025-11-28 16:25:00
Bulan yang muncul: [np.int32(8), np.int32(9), np.int32(10), np.int32(11)]

Baris yang berbeda antara cara umum dan cara benar: 0


> **Pelajaran yang dibawa pulang:** setelah setiap konversi tanggal, jalankan satu uji akal sehat.
> Yang paling murah adalah memeriksa bulan atau tahun yang muncul:
>
> ```python
> print(sorted(kolom.dt.month.unique()))
> print(kolom.min(), kolom.max())
> ```
>
> Kalau rentangnya tidak sesuai dengan yang Anda ketahui tentang data itu, konversinya salah —
> berapa pun jumlah galat yang muncul.

### TUGAS 7 dan 12 — tipe data

In [36]:
# TUGAS 7
t7_rp = df['total_bayar'].astype(str).str.startswith('Rp').sum()   # berapa baris yang total_bayar-nya tersimpan dengan awalan "Rp"

cek('T7_rp', t7_rp)

BENAR   T7_rp = 250


In [37]:
# TUGAS 12
# Pakai hasil konversi yang BENAR (variabel benar dari CONTOH 7.4).
t = df['tanggal_transaksi'].astype(str)
iso = t.str.match(r'^\d{4}-')
benar = pd.Series(pd.NaT, index=t.index, dtype='datetime64[ns]')
benar[iso] = pd.to_datetime(t[iso], format='%Y-%m-%d %H:%M:%S', errors='coerce')
benar[~iso] = pd.to_datetime(t[~iso], format='%d/%m/%Y %H:%M', errors='coerce')

t12_bulan = benar.dt.month.dropna().nunique()   # berapa bulan berbeda yang muncul di data
t12_rusak = int((benar[iso].dt.day <= 12).sum())  # berapa baris yang nilainya berbeda antara cara umum dan cara benar

cek('T12_bulan', t12_bulan)
cek('T12_rusak', t12_rusak)

BENAR   T12_bulan = 4
BELUM   T12_rusak = 524
        Petunjuk: Bandingkan hasil konversi yang benar dengan hasil pd.to_datetime(..., format='mixed', dayfirst=True) memakai !=, lalu jumlahkan.


---
## Bagian 8 — Validitas dan akurasi: aturan yang harus dipenuhi data

Dua pemeriksaan terakhir tidak bisa ditebak dari datanya saja. Keduanya berasal dari **pengetahuan
tentang hal yang diukur**, dan karena itu harus Anda tuliskan sendiri sebagai daftar aturan.

In [38]:
# CONTOH 8.1 — aturan rentang
aturan = {
    'jumlah harus positif'          : (df['jumlah'] <= 0),
    'diskon harus 0-100'            : (df['diskon_persen'] < 0) | (df['diskon_persen'] > 100),
    'rating harus 1-5'              : (df['rating_layanan'] < 1) | (df['rating_layanan'] > 5),
    'harga satuan harus positif'    : (df['harga_satuan'] <= 0),
}

for nama, pelanggar in aturan.items():
    print(f'{nama:32s} dilanggar {int(pelanggar.sum()):4d} baris')

jumlah harus positif             dilanggar   29 baris
diskon harus 0-100               dilanggar    9 baris
rating harus 1-5                 dilanggar   49 baris
harga satuan harus positif       dilanggar    0 baris


In [39]:
# CONTOH 8.2 — aturan hubungan antarkolom
# total_bayar seharusnya = jumlah x harga_satuan x (1 - diskon/100)
# Hanya bisa diperiksa pada baris yang total_bayar-nya sudah berupa angka.
num = df[~df['total_bayar'].astype(str).str.startswith('Rp')].copy()
num['total_angka'] = pd.to_numeric(num['total_bayar'], errors='coerce')
num['seharusnya']  = (num['jumlah'] * num['harga_satuan'] * (1 - num['diskon_persen'] / 100)).round()

tidak_cocok = num['total_angka'] != num['seharusnya']
print('Baris yang bisa diperiksa :', len(num))
print('Tidak cocok               :', int(tidak_cocok.sum()))
print()
num.loc[tidak_cocok, ['jumlah', 'harga_satuan', 'diskon_persen', 'total_angka', 'seharusnya']].head(6)

Baris yang bisa diperiksa : 1293
Tidak cocok               : 84



,jumlah,harga_satuan,diskon_persen,total_angka,seharusnya
3,1,500,0,250,500.0
13,3,6000,0,36000,18000.0
36,5,6000,0,15000,30000.0
62,2,12000,0,12000,24000.0
72,2,10000,0,200000,20000.0
130,2,5000,-20,10000,12000.0


> **Perhatikan keterbatasannya.** Pemeriksaan ini hanya berjalan pada baris yang total_bayar-nya sudah
> berupa angka. Ratusan baris lain belum bisa diperiksa sampai kolomnya dikonversi pada pertemuan 5.
>
> Ini pola yang berulang: satu cacat menyembunyikan cacat lain. Karena itu diagnosis dilakukan
> **berlapis**, dan diulang setelah pembersihan.

### TUGAS 8 dan 10 — validitas dan aturan bisnis

In [40]:
# TUGAS 8
t8_jumlah = (df['jumlah'] <= 0).sum()   # berapa baris dengan jumlah <= 0
t8_diskon = ((df['diskon_persen'] < 0) | (df['diskon_persen'] > 100)).sum()   # berapa baris dengan diskon di luar rentang 0-100

cek('T8_jumlah', t8_jumlah)
cek('T8_diskon', t8_diskon)

BENAR   T8_jumlah = 29
BENAR   T8_diskon = 9


In [41]:
# TUGAS 10
t10_bisnis = (num['total_angka'] != num['seharusnya']).sum()  # berapa baris (dari yang total_bayar-nya numerik) yang melanggar aturan hitungan

cek('T10_bisnis', t10_bisnis)

BENAR   T10_bisnis = 84


---
## Bagian 9 — Merangkai temuan menjadi profil kualitas data

Pemeriksaan yang berserakan di banyak sel tidak berguna bagi siapa pun. Rangkum menjadi satu tabel
yang bisa dibaca orang lain dalam satu menit. Tabel inilah inti **M2**.

In [42]:
# CONTOH 9.1 — kerangka profil kualitas data
temuan = []

def catat(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan.append({
        'dimensi'   : dimensi,
        'kolom'     : kolom,
        'masalah'   : masalah,
        'jumlah'    : int(jumlah),
        'persen'    : round(jumlah / len(df) * 100, 2),
        'keparahan' : keparahan,          # Tinggi / Sedang / Rendah
        'rencana'   : rencana,            # diisi lengkap pada pertemuan 5
    })

catat('Kelengkapan', 'saldo_simpanan', 'Kosong, diduga MNAR',
      df['saldo_simpanan'].isna().sum(), 'Tinggi', 'Jangan diimputasi; tanyakan ke pengurus koperasi')
catat('Kelengkapan', 'berat_kg', 'Kosong untuk kategori non-fisik (MAR)',
      df['berat_kg'].isna().sum(), 'Rendah', 'Tandai tidak berlaku, bukan diisi')
catat('Kelengkapan', 'rating_layanan', 'Placeholder -999',
      (df['rating_layanan'] == -999).sum(), 'Tinggi', 'Ubah menjadi NaN sebelum dihitung')
catat('Keunikan', 'seluruh kolom', 'Baris duplikat persis',
      df.duplicated().sum(), 'Sedang', 'Hapus, simpan satu')
catat('Konsistensi', 'kategori', 'Satu kategori ditulis beberapa cara',
      df['kategori'].astype(str).nunique() - 5, 'Tinggi', 'Seragamkan; sinonim diputuskan manual')
catat('Validitas', 'jumlah', 'Nilai nol atau negatif',
      (df['jumlah'] <= 0).sum(), 'Tinggi', 'Telusuri apakah retur; bila bukan, keluarkan dari analisis')

profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
1,Kelengkapan,berat_kg,Kosong untuk kategori non-fisik (MAR),351,22.75,Rendah,"Tandai tidak berlaku, bukan diisi"
3,Keunikan,seluruh kolom,Baris duplikat persis,24,1.56,Sedang,"Hapus, simpan satu"
0,Kelengkapan,saldo_simpanan,"Kosong, diduga MNAR",159,10.30,Tinggi,Jangan diimputasi; tanyakan ke pengurus koperasi
2,Kelengkapan,rating_layanan,Placeholder -999,49,3.18,Tinggi,Ubah menjadi NaN sebelum dihitung
5,Validitas,jumlah,Nilai nol atau negatif,29,1.88,Tinggi,"Telusuri apakah retur; bila bukan, keluarkan d..."
4,Konsistensi,kategori,Satu kategori ditulis beberapa cara,13,0.84,Tinggi,Seragamkan; sinonim diputuskan manual


### TUGAS 11 — lengkapi profilnya

Tambahkan **minimal empat temuan lagi** dengan memanggil `catat(...)`, lalu tampilkan ulang tabelnya.
Kandidat yang belum masuk: placeholder teks pada `metode_bayar`, `total_bayar` bertipe teks,
format tanggal campur, spasi pada `id_anggota`, diskon di luar rentang, pelanggaran aturan hitungan,
dan `id_transaksi` yang kosong atau berulang.

Tidak ada sel `cek` untuk tugas ini — penilaiannya memakai Rubrik M2.

In [43]:
# TUGAS 11
# Tambahan minimal 4 temuan baru sesuai instruksi modul
catat(
    'Tipe Data',
    'total_bayar',
    'String berformat mata uang Rp',
    t7_rp,
    'Tinggi',
    'Bersihkan karakter teks Rp dan titik, konversi ke integer/float',
)
catat(
    'Tipe Data',
    'tanggal_transaksi',
    'Format campuran ISO dan DD/MM/YYYY',
    (~iso).sum(),
    'Tinggi',
    'Parsing terpisah per format regex untuk menghindari bulan tertukar',
)
catat(
    'Konsistensi',
    'id_anggota',
    'Trailing whitespace (anggota hantu)',
    t9_anggota - t9_strip,
    'Tinggi',
    'Terapkan .str.strip() pada primary key transaksi',
)
catat(
    'Validitas',
    'diskon_persen',
    'Persentase di luar rentang valid 0-100',
    t8_diskon,
    'Tinggi',
    'Koreksi nilai negatif dan batasi nilai >100%',
)
catat(
    'Akurasi',
    'total_bayar',
    'Kalkulasi tidak sesuai rumus harga x jumlah',
    t10_bisnis,
    'Sedang',
    'Hitung ulang formula total_bayar setelah data numerik dibersihkan',
)
catat(
    'Keunikan',
    'id_transaksi',
    'ID transaksi berulang (duplikat nyaris)',
    t5_dupid - t5_dup,
    'Tinggi',
    'Rekonsiliasi variasi teks kasir dan kolom catatan',
)

profil = pd.DataFrame(temuan).sort_values(
    ['keparahan', 'jumlah'], ascending=[True, False]
)
profil


profil = pd.DataFrame(temuan).sort_values(['keparahan', 'jumlah'], ascending=[True, False])
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
1,Kelengkapan,berat_kg,Kosong untuk kategori non-fisik (MAR),351,22.75,Rendah,"Tandai tidak berlaku, bukan diisi"
10,Akurasi,total_bayar,Kalkulasi tidak sesuai rumus harga x jumlah,84,5.44,Sedang,Hitung ulang formula total_bayar setelah data ...
3,Keunikan,seluruh kolom,Baris duplikat persis,24,1.56,Sedang,"Hapus, simpan satu"
7,Tipe Data,tanggal_transaksi,Format campuran ISO dan DD/MM/YYYY,307,19.90,Tinggi,Parsing terpisah per format regex untuk menghi...
6,Tipe Data,total_bayar,String berformat mata uang Rp,250,16.20,Tinggi,"Bersihkan karakter teks Rp dan titik, konversi..."
0,Kelengkapan,saldo_simpanan,"Kosong, diduga MNAR",159,10.30,Tinggi,Jangan diimputasi; tanyakan ke pengurus koperasi
11,Keunikan,id_transaksi,ID transaksi berulang (duplikat nyaris),68,4.41,Tinggi,Rekonsiliasi variasi teks kasir dan kolom catatan
8,Konsistensi,id_anggota,Trailing whitespace (anggota hantu),59,3.82,Tinggi,Terapkan .str.strip() pada primary key transaksi
2,Kelengkapan,rating_layanan,Placeholder -999,49,3.18,Tinggi,Ubah menjadi NaN sebelum dihitung
5,Validitas,jumlah,Nilai nol atau negatif,29,1.88,Tinggi,"Telusuri apakah retur; bila bukan, keluarkan d..."


In [44]:
# Simpan profil Anda
profil.to_csv('profil_kualitas_latihan.csv', index=False)
print('Tersimpan:', profil.shape)

Tersimpan: (12, 7)


---
## Bagian 10 — Terapkan pada data proyek tim

**Ini bagian yang dinilai.** Hasilnya menjadi bagian pertama dari **M2 — Data Quality Report**
yang dikumpulkan pada pertemuan 5.

Kerjakan bersama tim pada dataset proyek Anda sendiri.

In [45]:
# 10.1 — muat dataset proyek tim
# df_tim = pd.read_csv('/content/.....csv')
# df_tim.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 379130 entries, 0 to 379129
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionNo  379130 non-null  object 
 1   Date           379130 non-null  object 
 2   ProductNo      379130 non-null  object 
 3   ProductName    379130 non-null  object 
 4   Price          379130 non-null  float64
 5   Quantity       379129 non-null  float64
 6   CustomerNo     379092 non-null  float64
 7   Country        379129 non-null  object 
dtypes: float64(3), object(5)
memory usage: 23.1+ MB


In [46]:
# 10.1 — muat dataset proyek tim
df_tim = pd.read_csv('/content/Sales Transaction v.4a.csv')
df_tim.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 536350 entries, 0 to 536349
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionNo  536350 non-null  object 
 1   Date           536350 non-null  object 
 2   ProductNo      536350 non-null  object 
 3   ProductName    536350 non-null  object 
 4   Price          536350 non-null  float64
 5   Quantity       536350 non-null  int64  
 6   CustomerNo     536295 non-null  float64
 7   Country        536350 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.7+ MB


In [ ]:
# 10.2 — jalankan kesembilan pemeriksaan pada data tim
# Salin pola Bagian 2 sampai 8. Untuk setiap pemeriksaan, catat temuannya.
# 1. Potret awal: shape, info, describe
# 2. Nilai kosong sejati
# 3. Placeholder teks dan angka
# 4. Pola kekosongan: MCAR / MAR / MNAR untuk setiap kolom yang banyak kosong
# 5. Duplikat persis dan duplikat menurut kunci
# 6. Konsistensi kolom teks
# 7. Tipe data yang salah
# 8. Aturan rentang
# 9. Aturan hubungan antarkolom


In [47]:
# 10.2 — Jalankan kesembilan pemeriksaan pada data tim

# 1. Potret Awal
print('=== 1. POTRET AWAL ===')
print('Dimensi:', df_tim.shape)
print('\nRingkasan Statistik Numerik:')
print(df_tim.describe().round(2))
print('\nRingkasan Kolom Kategori/Teks:')
print(df_tim.describe(include='object').T)

# 2. Nilai Kosong Sejati (NaN)
print('\n=== 2. NILAI KOSONG SEJATI ===')
kosong = df_tim.isna().sum()
print(
    kosong[kosong > 0].to_frame('jumlah_kosong').assign(
        persen=lambda d: (d.jumlah_kosong / len(df_tim) * 100).round(4)
    )
)

# 3. Placeholder Teks dan Angka
print('\n=== 3. PLACEHOLDER TEKS & ANGKA ===')
CURIGA = [
    'n/a',
    'na',
    '-',
    '',
    'null',
    'none',
    'tidak diketahui',
    'unknown',
    '?',
    '#n/a',
]
for kol in df_tim.select_dtypes(include='object').columns:
  nilai = df_tim[kol].astype(str).str.strip().str.lower()
  ketemu = nilai[nilai.isin(CURIGA)].value_counts()
  if len(ketemu):
    print(f'Placeholder teks pada {kol}:', ketemu.to_dict())

# 4. Pola Kekosongan (MCAR / MAR / MNAR)
# Kolom yang memiliki nilai kosong adalah CustomerNo (55 baris / 0.01%)
print('\n=== 4. ANALISIS POLA KEKOSONGAN (CustomerNo) ===')
m_cust = df_tim['CustomerNo'].isna()
trans_retur_cust = df_tim.loc[m_cust, 'TransactionNo'].str.startswith('C').sum()
print(
    f'CustomerNo kosong pada transaksi pembatalan (kode C): {trans_retur_cust} dari {m_cust.sum()} baris ({round(trans_retur_cust/m_cust.sum()*100, 1)}%)'
)
# Penjelasan: Pola ini tergolong MAR (Missing at Random) karena kekosongan CustomerNo
# secara kuat berhubungan dengan jenis transaksi retur/kredit (TransactionNo berawalan 'C')
# di mana kasir/sistem kasir tidak mencatat kembali identitas pelanggan saat barang dikembalikan.

# 5. Keunikan & Duplikasi
print('\n=== 5. KEUNIKAN DATA ===')
dup_persis = df_tim.duplicated().sum()
dup_item = df_tim.duplicated(
    subset=['TransactionNo', 'ProductNo'], keep=False
).sum()
print(f'Duplikat persis seluruh kolom : {dup_persis}')
print(f'Duplikat kombinasi transaksi & produk: {dup_item}')

# 6. Konsistensi Kolom Teks
print('\n=== 6. KONSISTENSI TEKS ===')
for kol in ['Country', 'ProductName', 'ProductNo', 'TransactionNo']:
  s = df_tim[kol].astype(str)
  mentah = s.nunique()
  rapi = s.str.strip().nunique()
  if mentah != rapi:
    print(f'{kol}: ada selisih spasi ({mentah - rapi} entitas redundan)')
  else:
    print(f'{kol}: konsisten (spasi aman, {mentah} nilai unik)')

# 7. Tipe Data yang Salah
print('\n=== 7. DIAGNOSIS TIPE DATA ===')
# Kolom Date bertipe object (string), seharusnya datetime
# Kolom CustomerNo bertipe float64 akibat NaN, seharusnya integer ID
print('Dtype Date      :', df_tim['Date'].dtype, '(Seharusnya Datetime)')
print('Dtype CustomerNo:', df_tim['CustomerNo'].dtype, '(Seharusnya Integer/ID)')

# 8. Aturan Rentang (Minimal 5 Aturan Validitas)
print('\n=== 8. ATURAN RENTANG ===')
aturan_rentang = {
    'Harga harus positif (> 0)': (df_tim['Price'] <= 0),
    'Kuantitas bernilai negatif (< 0)': (df_tim['Quantity'] < 0),
    'Kuantitas bernilai nol (== 0)': (df_tim['Quantity'] == 0),
    'CustomerNo dalam rentang ID valid (>= 10000)': (
        df_tim['CustomerNo'].dropna() < 10000
    ),
    'Negara tidak boleh Unspecified': (df_tim['Country'] == 'Unspecified'),
}
for nama, kondisi in aturan_rentang.items():
  print(f'{nama:45s}: dilanggar {int(kondisi.sum())} baris')

# 9. Aturan Hubungan Antarkolom (Cross-Column Rule)
print('\n=== 9. ATURAN HUBUNGAN ANTARKOLOM ===')
# Aturan bisnis: Jika kode transaksi berawalan 'C' (Cancellation), maka Quantity HARUS negatif
is_c = df_tim['TransactionNo'].astype(str).str.startswith('C')
is_neg = df_tim['Quantity'] < 0
pelanggaran_retur = (is_c != is_neg).sum()
print(
    f"Inkonsistensi status 'C' vs Quantity negatif: {pelanggaran_retur} baris"
)

=== 1. POTRET AWAL ===
Dimensi: (536350, 8)

Ringkasan Statistik Numerik:
           Price   Quantity  CustomerNo
count  536350.00  536350.00   536295.00
mean       12.66       9.92    15227.89
std         8.49     216.66     1716.58
min         5.13  -80995.00    12004.00
25%        10.99       1.00    13807.00
50%        11.94       3.00    15152.00
75%        14.09      10.00    16729.00
max       660.62   80995.00    18287.00

Ringkasan Kolom Kategori/Teks:
                count unique                                 top    freq
TransactionNo  536350  23204                              573585    1111
Date           536350    305                           12/5/2019    5299
ProductNo      536350   3768                              85123A    2378
ProductName    536350   3768  Cream Hanging Heart T-Light Holder    2378
Country        536350     38                      United Kingdom  485095

=== 2. NILAI KOSONG SEJATI ===
            jumlah_kosong  persen
CustomerNo             55  0.0

In [ ]:
# 10.3 — susun profil kualitas data tim
# Pakai fungsi catat() yang sama, lalu simpan.
# temuan_tim = []
# ...
# profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False)


In [48]:
# 10.3 — Susun profil kualitas data tim
temuan_tim = []


def catat_tim(dimensi, kolom, masalah, jumlah, keparahan, rencana):
  temuan_tim.append({
      'dimensi': dimensi,
      'kolom': kolom,
      'masalah': masalah,
      'jumlah': int(jumlah),
      'persen': round(jumlah / len(df_tim) * 100, 2),
      'keparahan': keparahan,
      'rencana': rencana,
  })


# 1. Kelengkapan
catat_tim(
    'Kelengkapan',
    'CustomerNo',
    'Nilai kosong sejati (NaN), 98% terjadi pada transaksi retur',
    df_tim['CustomerNo'].isna().sum(),
    'Sedang',
    'Tetapkan sebagai Guest/Non-Member, jangan diimputasi mean/median',
)

# 2. Keunikan
catat_tim(
    'Keunikan',
    'seluruh kolom',
    'Baris duplikat persis di seluruh variabel transaksi',
    df_tim.duplicated().sum(),
    'Sedang',
    'Hapus menggunakan drop_duplicates() dan simpan record pertama',
)

# 3. Validitas (Kuantitas Negatif)
catat_tim(
    'Validitas',
    'Quantity',
    'Kuantitas negatif akibat pembatalan pesanan (kode C)',
    (df_tim['Quantity'] < 0).sum(),
    'Tinggi',
    'Pisahkan transaksi retur ke tabel terpisah agar omzet tidak bias',
)

# 4. Tipe Data (Tanggal)
catat_tim(
    'Tipe Data',
    'Date',
    'Tersimpan sebagai teks (object), bukan tipe tanggal',
    len(df_tim),
    'Sedang',
    "Konversi ke datetime dengan format='%m/%d/%Y'",
)

# 5. Tipe Data (Customer ID)
catat_tim(
    'Tipe Data',
    'CustomerNo',
    'Tersimpan sebagai float64 akibat keberadaan NaN',
    (len(df_tim) - df_tim['CustomerNo'].isna().sum()),
    'Rendah',
    'Ubah ke nullable integer Int64 atau string identifikasi',
)

# 6. Konsistensi
catat_tim(
    'Konsistensi',
    'Country',
    'Terdapat entitas negara tidak spesifik (Unspecified)',
    (df_tim['Country'] == 'Unspecified').sum(),
    'Rendah',
    'Pertahankan kategori Unspecified pada analisis geospasial',
)

# 7. Akurasi & Integritas Hubungan
catat_tim(
    'Akurasi',
    'TransactionNo & Quantity',
    "Inkonsistensi antara awalan kode 'C' dan kuantitas negatif",
    pelanggaran_retur,
    'Rendah',
    'Korelasi 100% konsisten (0 pelanggaran), pertahankan filter retur',
)

profil_tim = pd.DataFrame(temuan_tim).sort_values(
    ['keparahan', 'jumlah'], ascending=[True, False]
)

# Buat folder reports jika belum ada
os.makedirs('reports', exist_ok=True)
profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False)

print('Profil kualitas data berhasil disimpan di reports/profil_kualitas_data.csv')
profil_tim

Profil kualitas data berhasil disimpan di reports/profil_kualitas_data.csv


,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
4,Tipe Data,CustomerNo,Tersimpan sebagai float64 akibat keberadaan NaN,536295,99.99,Rendah,Ubah ke nullable integer Int64 atau string ide...
5,Konsistensi,Country,Terdapat entitas negara tidak spesifik (Unspec...,447,0.08,Rendah,Pertahankan kategori Unspecified pada analisis...
6,Akurasi,TransactionNo & Quantity,Inkonsistensi antara awalan kode 'C' dan kuant...,0,0.00,Rendah,"Korelasi 100% konsisten (0 pelanggaran), perta..."
3,Tipe Data,Date,"Tersimpan sebagai teks (object), bukan tipe ta...",536350,100.00,Sedang,Konversi ke datetime dengan format='%m/%d/%Y'
1,Keunikan,seluruh kolom,Baris duplikat persis di seluruh variabel tran...,5200,0.97,Sedang,Hapus menggunakan drop_duplicates() dan simpan...
0,Kelengkapan,CustomerNo,"Nilai kosong sejati (NaN), 98% terjadi pada tr...",55,0.01,Sedang,"Tetapkan sebagai Guest/Non-Member, jangan diim..."
2,Validitas,Quantity,Kuantitas negatif akibat pembatalan pesanan (k...,8585,1.60,Tinggi,Pisahkan transaksi retur ke tabel terpisah aga...


### Daftar periksa Bagian 10

- [ ] Seluruh sembilan pemeriksaan dijalankan pada data tim
- [ ] Setiap kolom yang punya nilai kosong di atas 5% sudah dinilai MCAR, MAR, atau MNAR, **beserta alasannya**
- [ ] Minimal lima aturan validitas ditulis sendiri berdasarkan pengetahuan tentang data tim
- [ ] Minimal satu aturan hubungan antarkolom diperiksa
- [ ] Profil kualitas data tersimpan di `reports/`
- [ ] Notebook di-*commit* ke repositori tim dengan pesan yang bermakna

> **Ingat:** hari ini **tidak ada** yang diperbaiki. Perbaikan beserta justifikasinya adalah pekerjaan
> pertemuan 5. Yang diserahkan hari ini adalah diagnosis.

---
## Kalau Anda mentok

Ikuti urutan ini sebelum bertanya:

| Urutan | Yang dilakukan |
|---|---|
| 1 | Baca pesan galat **baris terakhirnya** — di situ nama galat dan penyebabnya |
| 2 | Jalankan ulang dari sel paling atas: *Runtime → Restart and run all* |
| 3 | Periksa kembali petunjuk yang muncul dari sel `cek(...)` |
| 4 | Tanyakan ke anggota tim; kemungkinan besar mereka menemui galat yang sama |
| 5 | Tanyakan ke asisten saat sesi berlangsung |
| 6 | Tulis di **forum tanya-jawab LMS**. Dosen menjawab setiap hari kerja. |

**Cara bertanya yang membuat jawaban datang lebih cepat:** sebutkan nomor bagian, tempelkan baris
terakhir pesan galat, dan sebutkan apa yang sudah Anda coba. Pertanyaan "kodenya error" tanpa keterangan
akan dibalas dengan pertanyaan balik, dan Anda kehilangan satu hari.

---
## Ringkasan pertemuan 4

1. **Lima dimensi:** kelengkapan, keunikan, konsistensi, validitas, akurasi. Periksa berurutan.
2. **Placeholder lolos dari `isna()`.** `-999` dan `N/A` terlihat seperti data sungguhan.
3. **MAR dapat dibuktikan; MCAR dan MNAR tidak.** Membedakannya butuh pengetahuan tentang cara data dikumpulkan.
4. **Duplikat nyaris lebih berbahaya daripada duplikat persis**, karena `duplicated()` tidak menangkapnya.
5. **Spasi tersembunyi menciptakan entitas hantu** dan diam-diam merusak hasil agregasi.
5b. **Konversi tanggal bisa salah tanpa satu pun galat.** Selalu periksa rentang dan bulan hasilnya.
6. **Satu cacat menyembunyikan cacat lain.** Diagnosis diulang setelah pembersihan.
7. **Hari ini mendiagnosis, belum memperbaiki.**

### Yang dikerjakan sebelum pertemuan 5

- Selesaikan Bagian 10 bersama tim dan *commit* ke repositori.
- Kerjakan kuis formatif pertemuan 4 di LMS (boleh diulang sebanyak yang Anda mau).
- Baca Bagian 1 workbook pertemuan 5 agar sesi berikutnya bisa langsung praktik.